# Train the Transformer on Colab (T13)

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. The repo is cloned into your Google Drive, so checkpoints and results survive a disconnect.
   If Colab disconnects, run all cells again: finished runs are skipped.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Clone the repo into Drive (or update it if it is already there), install it and build the token ids.

In [ ]:
REPO = "/content/drive/MyDrive/language-models-from-scratch"
!test -d {REPO} || git clone https://github.com/mohamadalichamseddine/language-models-from-scratch.git {REPO}
%cd {REPO}
!git pull
!pip install -q -r requirements.txt
!python scripts/prepare_data.py

Run the tests, then a 30-step smoke run to check that training works on the GPU.
Its output shows how many seconds 10 steps take (a full run is at most 4,000 steps).

In [ ]:
!python -m pytest -q
!python scripts/train_transformer.py --run smoke --max-steps 30 --eval-every 10 --eval-tokens 20000 --results-dir /content/smoke --checkpoint-dir /content/smoke

## The sweep

Trains and scores 9 runs at context 128: learning rate 3e-4, 6e-4, 1e-3 × dropout 0.1, 0.2, 0.3.
It writes `results/transformer/sweep/`, the frozen best settings in `configs/transformer.json`,
and the comparison table `results/comparison.md`.

In [ ]:
!python scripts/sweep_transformer.py

In [ ]:
!cat results/transformer/sweep/sweep.md results/comparison.md configs/transformer.json

## Download the results

The zip holds the run folders (without the large `.npz` prediction files), the tables and the
frozen config. Unzip it in the repo root on your laptop and commit. The checkpoints stay in Drive,
in `checkpoints/transformer/sweep/` (the best one is needed for T14 and T17).

In [ ]:
!zip -r /content/t13_results.zip results/transformer results/comparison.md results/comparison.csv configs/transformer.json -x "*.npz"
from google.colab import files
files.download("/content/t13_results.zip")